# Chromatic Point Cloud and Compression-Grid Bifiltration Visualization

This notebook visualizes the chromatic four-circle point-cloud example and its function-Alpha scc2020 chain-complex generators on the compressed finite grid. It does not call 2pac or GUDHI.

## 1. Imports and Paths

In [ ]:
from pathlib import Path
from pprint import pprint

from bifiltration_visualization import (
    describe_pointcloud,
    discover_pointcloud_files,
    discover_scc2020_files,
    generator_records,
    load_compressed_chain_complex,
    load_pointcloud_for_visualization,
    plot_pointcloud,
    reconstruct_geometric_simplices,
    summarize_compressed_complex,
)
from matplotlib.collections import LineCollection, PolyCollection
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
POINTCLOUD_DIR = PROJECT_ROOT / "data" / "pointcloud_examples"
SCC_DIR = PROJECT_ROOT / "data" / "chain_complex_scc2020"

CHROMATIC_STYLES = {
    0.0: {
        "label": "noisy circle points, f=0",
        "color": "#3b4dff",
        "edgecolor": "#1f2fbf",
        "marker": "o",
    },
    1.0: {
        "label": "sparse outliers, f=1",
        "color": "#ff914d",
        "edgecolor": "#d96a1e",
        "marker": "s",
    },
}


def _format_grid_value(value: float) -> str:
    return f"{float(value):.4g}"


def _chromatic_point_mask(function_values: np.ndarray, target: float) -> np.ndarray:
    return np.isclose(function_values, target)


def _scatter_chromatic_subset(
    ax,
    point_array: np.ndarray,
    function_values: np.ndarray,
    indices: np.ndarray,
    *,
    s: float,
    alpha: float,
    zorder: int,
    label_counts: bool = False,
) -> None:
    if len(indices) == 0:
        return

    selected_values = function_values[indices]
    used = np.zeros(len(indices), dtype=bool)
    for value, style in CHROMATIC_STYLES.items():
        mask = _chromatic_point_mask(selected_values, value)
        if not np.any(mask):
            continue
        selected_indices = indices[mask]
        label = style["label"]
        if label_counts:
            label = f"{label} ({len(selected_indices)})"
        ax.scatter(
            point_array[selected_indices, 0],
            point_array[selected_indices, 1],
            s=s,
            marker=style["marker"],
            color=style["color"],
            edgecolor=style["edgecolor"],
            linewidth=0.35 if alpha >= 0.5 else 0.0,
            alpha=alpha,
            label=label,
            zorder=zorder,
        )
        used |= mask

    if np.any(~used):
        selected_indices = indices[~used]
        ax.scatter(
            point_array[selected_indices, 0],
            point_array[selected_indices, 1],
            s=s,
            marker=".",
            color="0.35",
            edgecolors="none",
            alpha=alpha,
            label=f"other f values ({len(selected_indices)})" if label_counts else None,
            zorder=zorder,
        )


def plot_chromatic_pointcloud(pointcloud, *, title: str | None = None):
    if pointcloud.function_values is None:
        return plot_pointcloud(pointcloud, title=title)
    if pointcloud.points.shape[1] != 2:
        raise ValueError("Chromatic point-cloud plotting expects 2D points.")

    point_array = pointcloud.points
    function_values = np.asarray(pointcloud.function_values, dtype=float)
    fig, ax = plt.subplots(figsize=(6, 5))
    _scatter_chromatic_subset(
        ax,
        point_array,
        function_values,
        np.arange(pointcloud.point_count),
        s=26,
        alpha=0.9,
        zorder=2,
        label_counts=True,
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal", adjustable="box")
    ax.set_title(title or pointcloud.path.name)
    ax.legend(loc="best", frameon=True)
    fig.tight_layout()
    return fig, ax


def plot_chromatic_alpha_complex_at_grid(
    ax,
    pointcloud,
    simplex_records,
    grid_grade: tuple[int, int],
    *,
    show_all_points: bool = False,
    show_labels: bool = False,
) -> None:
    if pointcloud.function_values is None:
        raise ValueError("Chromatic alpha panels require point-cloud function values.")
    point_array = pointcloud.points
    function_values = np.asarray(pointcloud.function_values, dtype=float)
    if point_array.ndim != 2 or point_array.shape[1] != 2:
        raise ValueError("Chromatic alpha panels require 2D point coordinates.")

    present = [
        record
        for record in simplex_records
        if record.compressed_grade[0] <= grid_grade[0] and record.compressed_grade[1] <= grid_grade[1]
    ]
    triangles = [record.vertices for record in present if record.dimension == 2]
    edges = [record.vertices for record in present if record.dimension == 1]
    vertices = np.array(sorted({vertex for record in present for vertex in record.vertices}), dtype=int)

    if show_all_points:
        _scatter_chromatic_subset(
            ax,
            point_array,
            function_values,
            np.arange(pointcloud.point_count),
            s=6,
            alpha=0.22,
            zorder=1,
        )
    if triangles:
        polys = [point_array[list(triangle), :2] for triangle in triangles]
        ax.add_collection(
            PolyCollection(polys, facecolors="#8fb6d9", edgecolors="none", alpha=0.18, zorder=2)
        )
    if edges:
        segments = [point_array[list(edge), :2] for edge in edges]
        ax.add_collection(LineCollection(segments, colors="#334155", linewidths=0.65, alpha=0.65, zorder=3))
    if len(vertices):
        _scatter_chromatic_subset(
            ax,
            point_array,
            function_values,
            vertices,
            s=12,
            alpha=0.95,
            zorder=4,
        )

    ax.set_title(f"{grid_grade}", fontsize=7, pad=1.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal", adjustable="box")
    x_min, x_max = float(np.min(point_array[:, 0])), float(np.max(point_array[:, 0]))
    y_min, y_max = float(np.min(point_array[:, 1])), float(np.max(point_array[:, 1]))
    x_pad = max((x_max - x_min) * 0.04, 1e-9)
    y_pad = max((y_max - y_min) * 0.04, 1e-9)
    ax.set_xlim(x_min - x_pad, x_max + x_pad)
    ax.set_ylim(y_min - y_pad, y_max + y_pad)
    if show_labels:
        ax.set_xlabel(f"x={grid_grade[0]}", fontsize=8)
        ax.set_ylabel(f"y={grid_grade[1]}", fontsize=8)


def plot_chromatic_alpha_complex_grid(
    pointcloud,
    simplex_records,
    compression,
    *,
    x_coordinates,
    y_coordinates,
    vertical_spacing: float | None = None,
    show_all_points: bool = False,
    title: str = "Chromatic alpha complex at selected compression-grid grades",
):
    x_values = list(x_coordinates)
    y_values = list(y_coordinates)
    if not x_values or not y_values:
        raise ValueError("At least one x and y grid coordinate is required.")

    columns = len(x_values)
    rows = len(y_values)
    fig, axes = plt.subplots(rows, columns, figsize=(2.2 * columns, 2.55 * rows), squeeze=False)
    for row_index, y_value in enumerate(reversed(y_values)):
        for column_index, x_value in enumerate(x_values):
            ax = axes[row_index][column_index]
            plot_chromatic_alpha_complex_at_grid(
                ax,
                pointcloud,
                simplex_records,
                (x_value, y_value),
                show_all_points=show_all_points,
                show_labels=False,
            )
            if row_index == rows - 1:
                ax.set_xlabel(f"x={x_value}\nalpha={_format_grid_value(compression.x_values[x_value])}", fontsize=8)
            if column_index == 0:
                ax.set_ylabel(f"y={y_value}\nf={_format_grid_value(compression.y_values[y_value])}", fontsize=8)

    hspace = 0.48 if vertical_spacing is None else vertical_spacing
    fig.suptitle(title, y=0.985, fontsize=13)
    fig.subplots_adjust(left=0.055, right=0.99, bottom=0.12, top=0.86, wspace=0.10, hspace=hspace)
    return fig, axes


def _simplex_filtration_key(record):
    # Faces must appear before cofaces at the same radius.
    return (float(record.original_grade[0]), record.dimension, record.vertices)


def _codimension_one_faces(vertices: tuple[int, ...]) -> list[tuple[int, ...]]:
    if len(vertices) <= 1:
        return []
    return [tuple(vertices[:index] + vertices[index + 1 :]) for index in range(len(vertices))]


def _radius_to_compressed_x(compression, radius: float) -> int:
    x_values = np.asarray(compression.x_values, dtype=float)
    return int(np.argmin(np.abs(x_values - float(radius))))


def h1_persistence_pairs_at_y(simplex_records, compression, *, y_value: float = 0.0, y_tol: float = 1e-12):
    """Compute finite H1 persistence pairs for the one-parameter slice f <= y_value."""

    filtered_records = sorted(
        (
            record
            for record in simplex_records
            if record.dimension <= 2 and float(record.original_grade[1]) <= y_value + y_tol
        ),
        key=_simplex_filtration_key,
    )
    index_by_vertices = {}
    reduced_columns: dict[int, set[int]] = {}
    low_to_column: dict[int, int] = {}
    pairs = []

    for column_index, record in enumerate(filtered_records):
        vertices = tuple(record.vertices)
        boundary = set()
        for face in _codimension_one_faces(vertices):
            if face not in index_by_vertices:
                raise ValueError(f"Missing face {face} before simplex {vertices} in the y={y_value} filtration.")
            boundary.add(index_by_vertices[face])

        index_by_vertices[vertices] = column_index
        column = set(boundary)
        while column:
            pivot = max(column)
            if pivot not in low_to_column:
                break
            column ^= reduced_columns[low_to_column[pivot]]

        if not column:
            continue

        pivot = max(column)
        low_to_column[pivot] = column_index
        reduced_columns[column_index] = column

        birth_record = filtered_records[pivot]
        death_record = record
        if birth_record.dimension != 1:
            continue

        birth_radius = float(birth_record.original_grade[0])
        death_radius = float(death_record.original_grade[0])
        persistence = death_radius - birth_radius
        if persistence < -y_tol:
            continue

        pairs.append(
            {
                "birth_radius": birth_radius,
                "death_radius": death_radius,
                "persistence": max(0.0, persistence),
                "birth_x": _radius_to_compressed_x(compression, birth_radius),
                "death_x": _radius_to_compressed_x(compression, death_radius),
                "birth_simplex": birth_record.vertices,
                "death_simplex": death_record.vertices,
            }
        )

    return sorted(pairs, key=lambda pair: (-pair["persistence"], pair["birth_radius"], pair["death_radius"]))


def selected_h1_critical_x_coordinates(simplex_records, compression, *, y_value: float = 0.0, pair_count: int = 5):
    pairs = h1_persistence_pairs_at_y(simplex_records, compression, y_value=y_value)
    top_pairs = pairs[:pair_count]
    x_coordinates = {0, compression.n}
    x_coordinates.update(pair["birth_x"] for pair in top_pairs)
    return sorted(x_coordinates), top_pairs, pairs


print("PROJECT_ROOT:", PROJECT_ROOT)
print("POINTCLOUD_DIR:", POINTCLOUD_DIR)
print("SCC_DIR:", SCC_DIR)

## 2. Manual Paired File Selection

Set `POINTCLOUD_PATH` and `SCC_PATH` as a matched pair. File discovery below is only a reference list.

In [ ]:
pointcloud_files = discover_pointcloud_files(POINTCLOUD_DIR)
scc_files = discover_scc2020_files(SCC_DIR)

print("Point-cloud files:")
for index, path in enumerate(pointcloud_files):
    print(f"  [{index}] {path.relative_to(PROJECT_ROOT)}")

print("\nscc2020 files:")
for index, path in enumerate(scc_files):
    print(f"  [{index}] {path.relative_to(PROJECT_ROOT)}")

# Manual paired selection. Change both paths together.
POINTCLOUD_PATH = POINTCLOUD_DIR / "chromatic_four_circles_500pts_function.txt"
SCC_PATH = SCC_DIR / "chromatic_four_circles_500pts_function.alpha.scc2020.txt"

print("\nSelected point cloud:", POINTCLOUD_PATH)
print("Selected scc2020:", SCC_PATH)

## 3. Point Cloud Visualization

This chromatic point cloud uses `xyf` rows. The preview draws `f = 0` noisy circle points in blue and `f = 1` sparse outliers in orange.

In [ ]:
POINTCLOUD_MODE = "xyf"

try:
    pointcloud = load_pointcloud_for_visualization(POINTCLOUD_PATH, mode=POINTCLOUD_MODE)
    pprint(describe_pointcloud(pointcloud))
    plot_chromatic_pointcloud(pointcloud, title="Chromatic four-circle point cloud")
    plt.show()
except Exception as exc:
    pointcloud = None
    print(f"Point-cloud visualization skipped: {exc}")

## 4. Load and Compress scc2020 Chain Complex

In [ ]:
try:
    compressed_data = load_compressed_chain_complex(SCC_PATH)
    records = generator_records(compressed_data.complex, compressed_data.compressed_complex)
    geometric_records = reconstruct_geometric_simplices(
        compressed_data.complex,
        compressed_data.compressed_complex,
    )
    summary = summarize_compressed_complex(compressed_data)
    x_values = summary["x_values"]
    compact_summary = {
        "path": summary["path"],
        "max_dim": summary["max_dim"],
        "term_sizes": summary["term_sizes"],
        "grid_upper": summary["grid_upper"],
        "x_value_count": len(x_values),
        "x_value_preview": x_values[:5] + ["..."] + x_values[-5:] if len(x_values) > 10 else x_values,
        "y_values": summary["y_values"],
    }
    pprint(compact_summary)
    print("\nReconstructed geometric generators:", len(geometric_records))
    if pointcloud is not None:
        c0_count = len(compressed_data.complex.term_grades[0])
        if pointcloud.point_count < c0_count:
            raise ValueError(
                f"The paired point cloud has {pointcloud.point_count} points, but C0 has {c0_count} generators."
            )
    print("\nValidation valid:", compressed_data.validation.is_valid)
    print("Validation warnings:", compressed_data.validation.warnings)
    print("Validation errors:", compressed_data.validation.errors)
    print("Final grade block:", compressed_data.validation.has_final_grade_block)
except Exception as exc:
    compressed_data = None
    records = []
    geometric_records = []
    print(f"scc2020 loading skipped: {exc}")

## 5. Select Compression Grid Grades

The full compression grid uses integer coordinates from `(0, 0)` to `compression.upper`. This chromatic notebook chooses the displayed radius/alpha coordinates from the `y = 0` one-parameter filtration: it always includes `x = 0` and the final radius, then adds the birth radii from the top finite H1 birth-death pairs, sorted by persistence. For the chromatic function coordinate, the y-axis should usually compress to the two values `f = 0` and `f = 1`. `SHOW_ALL_POINTCLOUD_POINTS = True` keeps the full chromatic point cloud visible as a pale background.

In [ ]:
H1_CRITICAL_Y_VALUE = 0.0
H1_CRITICAL_PAIR_COUNT = 5
GRID_VERTICAL_SPACING = None  # None uses the automatic compact default; try 0.04 or 0.12 manually.
SHOW_ALL_POINTCLOUD_POINTS = True  # Show the full chromatic point cloud as a pale background.

if compressed_data is not None:
    X_GRID_COORDINATES, top_h1_pairs, all_h1_pairs = selected_h1_critical_x_coordinates(
        geometric_records,
        compressed_data.compression,
        y_value=H1_CRITICAL_Y_VALUE,
        pair_count=H1_CRITICAL_PAIR_COUNT,
    )
    Y_GRID_COORDINATES = list(range(compressed_data.compression.m + 1))
    print("Compression upper:", compressed_data.compression.upper)
    print("x coordinates shown:", X_GRID_COORDINATES)
    print("y coordinates shown:", Y_GRID_COORDINATES)
    print("H1 critical slice y value:", H1_CRITICAL_Y_VALUE)
    print("finite H1 pairs found in y=0 slice:", len(all_h1_pairs))
    print("top H1 pairs used for birth-radius columns:")
    for rank, pair in enumerate(top_h1_pairs, start=1):
        print(
            f"  {rank}. birth r={pair['birth_radius']:.6g} (x={pair['birth_x']}), "
            f"death r={pair['death_radius']:.6g} (x={pair['death_x']}), "
            f"persistence={pair['persistence']:.6g}, "
            f"birth simplex={pair['birth_simplex']}, death simplex={pair['death_simplex']}"
        )
    print("vertical subplot spacing:", GRID_VERTICAL_SPACING if GRID_VERTICAL_SPACING is not None else "auto")
    print("show full point-cloud background:", SHOW_ALL_POINTCLOUD_POINTS)
    print("x axis is the compressed radius/alpha coordinate.")
    print("y axis is the compressed chromatic function-value coordinate f in {0, 1}.")
else:
    X_GRID_COORDINATES, Y_GRID_COORDINATES = [], []
    print("No compressed chain complex loaded.")

## 6. Alpha Complex Panels on the Compression Grid

Each panel corresponds to one compressed grade `(i, j)`. It draws every reconstructed simplex whose compressed birth grade is `<= (i, j)` coordinatewise. Present vertices keep the chromatic colors (`f = 0` blue, `f = 1` orange); neutral edges and filled triangles show the currently present alpha complex. Pale background dots show the full paired point cloud when `SHOW_ALL_POINTCLOUD_POINTS = True`.

In [ ]:
if compressed_data is None:
    print("No compressed chain complex loaded.")
elif pointcloud is None:
    print("No paired point cloud loaded.")
else:
    try:
        plot_chromatic_alpha_complex_grid(
            pointcloud,
            geometric_records,
            compressed_data.compression,
            x_coordinates=X_GRID_COORDINATES,
            y_coordinates=Y_GRID_COORDINATES,
            vertical_spacing=GRID_VERTICAL_SPACING,
            show_all_points=SHOW_ALL_POINTCLOUD_POINTS,
            title="Chromatic alpha complex at selected compression-grid grades",
        )
        plt.show()
    except Exception as exc:
        print(f"Alpha-complex grid visualization skipped: {exc}")

## 7. Single Selected Grade

Set `SELECTED_GRID_GRADE` below, then rerun this cell to inspect one grade at a larger size.

In [ ]:
SELECTED_GRID_GRADE = (
    1379,
    1,
) if compressed_data is not None else (0, 0)

if compressed_data is None:
    print("No compressed chain complex loaded.")
elif pointcloud is None:
    print("No paired point cloud loaded.")
else:
    try:
        fig, ax = plt.subplots(figsize=(6, 6))
        plot_chromatic_alpha_complex_at_grid(
            ax,
            pointcloud,
            geometric_records,
            SELECTED_GRID_GRADE,
            show_all_points=SHOW_ALL_POINTCLOUD_POINTS,
            show_labels=True,
        )
        ax.set_title(f"Chromatic alpha complex at compressed grade {SELECTED_GRID_GRADE}", pad=10)
        fig.tight_layout(pad=1.8)
        plt.show()

        present = [
            record
            for record in geometric_records
            if record.compressed_grade[0] <= SELECTED_GRID_GRADE[0]
            and record.compressed_grade[1] <= SELECTED_GRID_GRADE[1]
        ]
        by_dim = {dim: sum(1 for record in present if record.dimension == dim) for dim in sorted({r.dimension for r in present})}
        print("Present simplex counts by dimension:", by_dim)
    except Exception as exc:
        print(f"Selected-grade visualization skipped: {exc}")